In [0]:
import re
import unicodedata
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
%run "../../utilidades/lib_utilidades"

In [0]:
str_catalogo = 'kaggle'

# Dados de entrada
str_schema_bronze = 'bronze'
str_nome_tabela_bronze = 'chicago_crimes'
str_endereco_tabela_bronze = f'{str_catalogo}.{str_schema_bronze}.{str_nome_tabela_bronze}'

# Dados de saida
str_schema_silver = 'silver'
str_nome_tabela_silver = 'crimes_chicago'
str_endereco_tabela_silver = f'{str_catalogo}.{str_schema_silver}.{str_nome_tabela_silver}'

In [0]:
sdf_bronze = spark.read.table(str_endereco_tabela_bronze)
display(sdf_bronze.limit(10))
count = sdf_bronze.count()
print(f'Quantidade de registros: {count}')

In [0]:
sdf_silver = (
    sdf_bronze
    .select(
        F.col('ID').alias('idt_crime'),
        F.col('Case_Number').alias('cod_case'),
        F.to_timestamp(F.col('Date'), 'MM/dd/yyyy hh:mm:ss a').alias('dtm_crime'),
        F.col('Block').alias('des_block'),
        F.col('IUCR').alias('cod_iucr'),
        F.col('Primary_Type').alias('des_primary_type'),
        F.col('Description').alias('des_crime'),
        F.col('Location_Description').alias('des_location'),
        F.col('Arrest').alias('flg_arrest'),
        F.col('Domestic').alias('flg_domestic'),
        F.col('Beat').alias('idt_beat'),
        F.col('District').alias('idt_district'),
        F.col('Ward').alias('idt_ward'),
        F.col('Community_Area').alias('idt_community_area'),
        F.col('FBI_Code').alias('cod_fbi'),
        F.col('X_Coordinate').alias('cod_x_coord'),
        F.col('Y_Coordinate').alias('cod_y_coord'),
        F.col('Year').alias('num_year'),
        F.to_timestamp(F.col('Updated_On'), 'MM/dd/yyyy hh:mm:ss a').alias('dtm_crime_update'),
        F.col('Latitude').alias('cod_latitude'),
        F.col('Longitude').alias('cod_longitude'),
        F.col('Location').alias('cod_location')
    )
)

In [0]:
sdf_silver = (
    sdf_silver
    .drop('cod_iucr', 'idt_beat', 'idt_district', 'idt_ward', 'idt_community_area', 'cod_fbi', 'cod_x_coord', 'cod_y_coord')
    .withColumn(
        'des_crime', 
        normalize_text_udf(F.col('des_crime'))
    )
    .filter(
        (F.col('cod_location').isNotNull()) & (F.col('des_location').isNotNull()) & # Removendo colunas onde as informações de localidade são nulas
        (F.col('cod_latitude').between(41, 43)) & (F.col('cod_longitude').between(-88, -87)) # Removendo colunas fora do circulo de Chicago
    )
    .dropDuplicates(['idt_crime'])
)

In [0]:
(
    sdf_silver.write.format('delta')
    .option('mergeSchema', 'true')
    .mode('append')
    .saveAsTable(str_endereco_tabela_silver)
)

In [0]:
display(spark.sql(f"OPTIMIZE {str_endereco_tabela_silver} ZORDER BY des_crime"))

In [0]:
str_des_tabela = "Tabela contendo registros de crimes ocorridos em Chicago, com informações padronizadas sobre data, localização, tipo de crime, latitude, longitude e indicadores de prisão e violência doméstica."

lst_colunas_comentarios = {
    'idt_crime': 'Identificador único do crime',
    'cod_case': 'Número do caso policial',
    'dtm_crime': 'Data e hora do crime',
    'des_block': 'Endereço aproximado do local do crime',
    'des_primary_type': 'Tipo principal do crime',
    'des_crime': 'Descrição detalhada do crime',
    'des_location': 'Descrição do local onde ocorreu o crime',
    'flg_arrest': 'Indicador se houve prisão (True/False)',
    'flg_domestic': 'Indicador de violência doméstica (True/False)',
    'num_year': 'Ano de ocorrência do crime',
    'dtm_crime_update': 'Data e hora da última atualização do registro',
    'cod_latitude': 'Latitude do local do crime',
    'cod_longitude': 'Longitude do local do crime',
    'cod_location': 'Coordenadas geográficas do local do crime'
}

comentarTabelas(
    str_endereco_tabela = str_endereco_tabela_silver, 
    str_des_tabela = str_des_tabela, 
    lst_colunas_comentarios = lst_colunas_comentarios
)